# FFCx Tutorial: Code Generation and Analysis

This comprehensive notebook covers:
1. 📝 Defining variational forms in UFL
2. ⚙️ Compiling forms with FFCx
3. 🔍 Analyzing generated C code
4. 📊 Understanding optimization techniques

**Perfect for Google Colab!** Just run the cells in order.

---

## 🚀 Setup: Install Required Packages

In [ ]:
%%capture
# Install FEniCS components silently
!pip install fenics-ffcx fenics-basix fenics-ufl

In [ ]:
# Verify installation
import ffcx
import ffcx.main
import basix.ufl
import ufl
import re
import os
from pathlib import Path
from collections import defaultdict

print("✅ All packages installed successfully!")
print(f"   FFCx version: {getattr(ffcx, '__version__', 'installed')}")

---
## Part 1: Simple Example - Mass Matrix

We'll start with the simplest bilinear form:

$$a(u, v) = \int_\Omega u \cdot v \, dx$$

In [ ]:
# Create the example file
example1_code = '''import basix.ufl
import ufl

# Define mesh and function space
cell = "triangle"
c_el = basix.ufl.element("Lagrange", cell, 1, shape=(2,))
domain = ufl.Mesh(c_el)
el = basix.ufl.element("Lagrange", cell, 2)
V = ufl.FunctionSpace(domain, el)

# Trial and test functions
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Mass matrix bilinear form
a = ufl.inner(u, v) * ufl.dx

# Required for FFCx
forms = [a]
'''

with open('mass_matrix.py', 'w') as f:
    f.write(example1_code)

print("✅ Example 1: Mass Matrix")
print("   Form: a(u, v) = ∫ u·v dx")
print("   Element: P2 Lagrange on triangles")
print("   Local DOFs: 6 per triangle")

In [ ]:
# Compile with FFCx
print("⚙️  Compiling with FFCx...")
ffcx.main.main(["-o", ".", "mass_matrix.py"])

# Check generated files
if os.path.exists('mass_matrix.c'):
    c_size = os.path.getsize('mass_matrix.c')
    h_size = os.path.getsize('mass_matrix.h')
    print(f"\n✅ Compilation successful!")
    print(f"   mass_matrix.c: {c_size:,} bytes")
    print(f"   mass_matrix.h: {h_size:,} bytes")
else:
    print("❌ Compilation failed!")

### 🔍 Inspect Generated Code

In [ ]:
# Show header file
print("="*70)
print(" HEADER FILE (mass_matrix.h) - First 25 lines")
print("="*70)

with open('mass_matrix.h', 'r') as f:
    for i, line in enumerate(f):
        if i >= 25:
            break
        print(line, end='')

In [ ]:
# Show function signature
print("="*70)
print(" ASSEMBLY FUNCTION SIGNATURE")
print("="*70)

with open('mass_matrix.c', 'r') as f:
    content = f.read()

pattern = r'void tabulate_tensor[^{]+'
match = re.search(pattern, content)
if match:
    print(match.group(0))
    print("\n📌 Key parameters:")
    print("   - A: Output local tensor (matrix)")
    print("   - coordinate_dofs: Element vertex coordinates")
    print("   - w: Coefficient function values")
else:
    print("Function signature not found")

In [ ]:
# Show quadrature rules
print("="*70)
print(" QUADRATURE RULES")
print("="*70)

pattern = r'static const double weights_\w+\[(\d+)\]\s*=\s*\{([^}]+)\}'
matches = re.findall(pattern, content)

if matches:
    for i, (n_points, vals) in enumerate(matches[:1], 1):
        print(f"Quadrature rule {i}:")
        print(f"  Number of points: {n_points}")
        print(f"  Weights:")
        weights = [float(v.strip()) for v in vals.split(',')[:int(n_points)]]
        for j, w in enumerate(weights):
            print(f"    w[{j}] = {w:.15f}")
        print()
else:
    print("No quadrature rules found")

---
## Part 2: More Complex Example - Poisson Equation

$$-\nabla^2 u = f \quad \text{in } \Omega$$

Weak form:
- Bilinear: $a(u, v) = \int_\Omega \nabla u \cdot \nabla v \, dx$
- Linear: $L(v) = \int_\Omega f \cdot v \, dx$

In [ ]:
# Create Poisson example
example2_code = '''import basix.ufl
import ufl

cell = "triangle"
c_el = basix.ufl.element("Lagrange", cell, 1, shape=(2,))
domain = ufl.Mesh(c_el)
el = basix.ufl.element("Lagrange", cell, 1)
V = ufl.FunctionSpace(domain, el)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
f = ufl.Coefficient(V)

# Poisson equation forms
a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = f * v * ufl.dx

forms = [a, L]
'''

with open('poisson.py', 'w') as f:
    f.write(example2_code)

print("✅ Example 2: Poisson Equation")
print("   Bilinear: a(u, v) = ∫ ∇u·∇v dx")
print("   Linear: L(v) = ∫ f·v dx")
print("   Element: P1 Lagrange")

In [ ]:
# Compile Poisson
print("⚙️  Compiling Poisson equation...")
ffcx.main.main(["-o", ".", "poisson.py"])

if os.path.exists('poisson.c'):
    print("\n✅ Compilation successful!")
else:
    print("❌ Compilation failed!")

---
## Part 3: Code Analysis

Let's analyze the generated code in detail.

In [ ]:
def analyze_file(filename):
    """Analyze a C file."""
    if not os.path.exists(filename):
        print(f"File not found: {filename}")
        return
    
    with open(filename, 'r') as f:
        lines = f.readlines()
        content = ''.join(lines)
    
    # Basic statistics
    total_lines = len(lines)
    code_lines = sum(1 for line in lines if line.strip() and not line.strip().startswith('//'))
    
    # Find patterns
    num_functions = content.count('void tabulate_tensor')
    for_pattern = r'for\s*\('
    num_loops = len(re.findall(for_pattern, content))
    num_static = content.count('static const double')
    
    # Quadrature
    quad_pattern = r'weights_\w+\[(\d+)\]'
    quad_match = re.search(quad_pattern, content)
    num_quad = int(quad_match.group(1)) if quad_match else 0
    
    print(f"\n📊 Analysis of {filename}:")
    print(f"   Total lines:          {total_lines:,}")
    print(f"   Code lines:           {code_lines:,}")
    print(f"   Assembly functions:   {num_functions}")
    print(f"   For loops:            {num_loops}")
    print(f"   Static arrays:        {num_static}")
    print(f"   Quadrature points:    {num_quad}")

print("="*70)
print(" CODE ANALYSIS")
print("="*70)

analyze_file('mass_matrix.c')
analyze_file('poisson.c')

In [ ]:
# Compare the two examples
print("="*70)
print(" COMPARISON")
print("="*70)

files = {
    'Mass Matrix (P2)': 'mass_matrix.c',
    'Poisson (P1)': 'poisson.c'
}

print(f"\n{'Example':<20} {'Lines':<10} {'Size (KB)':<10}")
print("-" * 40)

for name, fname in files.items():
    if os.path.exists(fname):
        with open(fname, 'r') as f:
            lines = len(f.readlines())
        size = os.path.getsize(fname) / 1024
        print(f"{name:<20} {lines:<10,} {size:<10.1f}")

print("\n💡 Observations:")
print("   - P2 elements have more DOFs (6 vs 3)")
print("   - Gradient operations add complexity")
print("   - More quadrature points for higher accuracy")

---
## Part 4: Deep Dive - Loop Structure

Let's examine the actual assembly loop in detail.

In [ ]:
print("="*70)
print(" ASSEMBLY LOOP STRUCTURE")
print("="*70)

with open('mass_matrix.c', 'r') as f:
    content = f.read()

# Find the main assembly loop
pattern = r'for \(int iq = 0.*?\n\s*\}\s*\n\s*\}'
match = re.search(pattern, content, re.DOTALL)

if match:
    loop_code = match.group(0)
    # Show first 500 characters
    print(loop_code[:500])
    print("\n... (truncated)")
    
    print("\n📌 Loop structure:")
    print("   1. Outer loop: quadrature points (iq)")
    print("   2. Inner loops: basis functions (i, j)")
    print("   3. Accumulate: A[i,j] += phi_i * phi_j * weight")
else:
    print("Loop not found")

---
## Part 5: Advanced - Multiple Examples

Let's create several examples at once.

In [ ]:
# Create file with multiple forms
multi_code = '''import basix.ufl
import ufl

cell = "triangle"
c_el = basix.ufl.element("Lagrange", cell, 1, shape=(2,))
domain = ufl.Mesh(c_el)

# P1 scalar
el1 = basix.ufl.element("Lagrange", cell, 1)
V1 = ufl.FunctionSpace(domain, el1)
u1 = ufl.TrialFunction(V1)
v1 = ufl.TestFunction(V1)

# P2 scalar
el2 = basix.ufl.element("Lagrange", cell, 2)
V2 = ufl.FunctionSpace(domain, el2)
u2 = ufl.TrialFunction(V2)
v2 = ufl.TestFunction(V2)

# Vector P1
el_vec = basix.ufl.element("Lagrange", cell, 1, shape=(2,))
V_vec = ufl.FunctionSpace(domain, el_vec)
u_vec = ufl.TrialFunction(V_vec)
v_vec = ufl.TestFunction(V_vec)

# Different forms
mass_p1 = ufl.inner(u1, v1) * ufl.dx
mass_p2 = ufl.inner(u2, v2) * ufl.dx
stiff_p1 = ufl.inner(ufl.grad(u1), ufl.grad(v1)) * ufl.dx
stiff_p2 = ufl.inner(ufl.grad(u2), ufl.grad(v2)) * ufl.dx
vec_laplacian = ufl.inner(ufl.grad(u_vec), ufl.grad(v_vec)) * ufl.dx

forms = [mass_p1, mass_p2, stiff_p1, stiff_p2, vec_laplacian]
'''

with open('multiple_forms.py', 'w') as f:
    f.write(multi_code)

print("✅ Created file with 5 different forms:")
print("   1. Mass matrix P1")
print("   2. Mass matrix P2")
print("   3. Stiffness matrix P1")
print("   4. Stiffness matrix P2")
print("   5. Vector Laplacian")

print("\n⚙️  Compiling all forms...")
ffcx.main.main(["-o", ".", "multiple_forms.py"])
print("✅ Done!")

In [ ]:
# Analyze the multi-form file
if os.path.exists('multiple_forms.c'):
    with open('multiple_forms.c', 'r') as f:
        content = f.read()
        lines = f.readlines()
    
    num_functions = content.count('void tabulate_tensor')
    file_size = os.path.getsize('multiple_forms.c')
    
    print("="*70)
    print(" MULTIPLE FORMS ANALYSIS")
    print("="*70)
    print(f"\n   Assembly functions: {num_functions}")
    print(f"   File size: {file_size:,} bytes ({file_size/1024:.1f} KB)")
    print(f"   Total lines: {len(content.splitlines()):,}")
    
    print("\n💡 FFCx generated optimized code for all 5 forms!")

---
## 📚 Summary

### What We Learned:

1. ✅ **UFL Forms** - Symbolic representation of variational problems
2. ✅ **FFCx Compilation** - Automatic C code generation
3. ✅ **Generated Code** - Optimized assembly kernels
4. ✅ **Analysis** - Understanding the compilation process

### Key Insights:

- **Automation**: FFCx eliminates manual assembly code
- **Optimization**: Precomputed basis functions, efficient loops
- **Flexibility**: Works with any UFL-expressible form
- **Performance**: Production-ready optimized code

### Generated Files:

Check your Colab files panel for:
- `mass_matrix.c/.h` - Simple mass matrix
- `poisson.c/.h` - Laplacian problem
- `multiple_forms.c/.h` - Multiple forms in one file

### Next Steps:

1. 🔬 Study the generated C code in detail
2. 🧪 Experiment with different element types
3. 🚀 Use DOLFINx to solve actual PDEs
4. 📖 Read FFCx documentation

### Resources:

- [FFCx GitHub](https://github.com/FEniCS/ffcx)
- [FEniCS Tutorial](https://jsdokken.com/dolfinx-tutorial/)
- [UFL Documentation](https://fenics.readthedocs.io/projects/ufl/)

---

**Happy coding! 🎉**